# Your first honest model with scikit-learn

Data Scientist track, lesson DS2. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** fit a first model on a training part, score it on held-out people beside two simpler rules, put an interval on the gain, find the columns that carry it, and write the verdict with a limitations line.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 2026       # the seed of the whole Data Scientist track: every split and every resample starts from it

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)

# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")

## Unit 1: Two rungs below the model

**Predict first.** Rung 1 gives everyone the same chance, 0.544. Rung 2 learns from channel, platform and region. How many more people out of 100 does rung 2 call right than rung 1: about 1, about 7 or about 20?

In [ ]:
STATIC = ["channel", "platform", "region"]                    # known at signup
WEEK1 = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d", "onboarded", "saw_paywall", "bought_plus"]    # known at day 7
X, y = table[STATIC + WEEK1], table["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
print(len(X_train), len(X_test), round(float(y_train.mean()), 4))

rung1 = DummyClassifier(strategy="prior").fit(X_train, y_train)     # the flat guess: everyone gets the training share of churners
rung2 = make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC)]), LogisticRegression(max_iter=1000))
rung2.fit(X_train, y_train)                                         # the day-0 rule: the three signup columns only
for name, m in [("rung 1", rung1), ("rung 2", rung2)]:
    print(name, round(accuracy_score(y_test, m.predict(X_test)), 3), round(roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]), 3))

## Unit 2: One object, three verbs

**Predict first.** `model.score(X_test, y_test)` is one number. Which is it: the accuracy, the ROC AUC or the share of churners? And is `predict` the same as asking whether the chance is above 0.5?

In [ ]:
model = make_pipeline(
    ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC),      # text columns: one 0/1 column per value
                       ("num", StandardScaler(), WEEK1)]),                           # number columns: centered and scaled
    LogisticRegression(max_iter=1000),
)
model.fit(X_train, y_train)                       # learns from the training part only
chance = model.predict_proba(X_test)[:, 1]        # a chance of churning for every test person
call = model.predict(X_test)                      # a 0 or a 1 for every test person
print(np.round(chance[:3], 3), call[:3])
print(round(model.score(X_test, y_test), 4), round(accuracy_score(y_test, call), 4), round(roc_auc_score(y_test, chance), 4))
print(bool(((chance > 0.5).astype(int) == call).all()))

## Unit 3: An interval on the gain

**Predict first.** The two models were scored on the same 4,454 people. Will the 95 percent interval of the gain in ROC AUC include 0, and about how wide will it be?

In [ ]:
rng = np.random.default_rng(SEED)
y_arr = y_test.to_numpy()
p_day0 = rung2.predict_proba(X_test)[:, 1]
gain = []
for _ in range(2000):
    i = rng.integers(0, len(y_arr), len(y_arr))                # one resample of the test people, used for both models
    gain.append(roc_auc_score(y_arr[i], chance[i]) - roc_auc_score(y_arr[i], p_day0[i]))
print(round(roc_auc_score(y_arr, chance) - roc_auc_score(y_arr, p_day0), 3))
print(np.round(np.percentile(gain, [2.5, 97.5]), 3))

## Unit 4: Which columns carry the gain

**Predict first.** Which removal costs more ROC AUC: the week-1 columns or the signup columns? And what do the three flags add?

In [ ]:
COUNTS, FLAGS = WEEK1[:4], WEEK1[4:]


def auc_with(cat, num):
    """Fit the same recipe on some of the columns and give its test AUC."""
    parts = ([("cat", OneHotEncoder(handle_unknown="ignore"), cat)] if cat else []) + ([("num", StandardScaler(), num)] if num else [])
    m = make_pipeline(ColumnTransformer(parts), LogisticRegression(max_iter=1000)).fit(X_train[cat + num], y_train)
    return round(float(roc_auc_score(y_test, m.predict_proba(X_test[cat + num])[:, 1])), 3)


print({"all columns": auc_with(STATIC, WEEK1), "without the week-1 columns": auc_with(STATIC, []),
       "without the signup columns": auc_with([], WEEK1), "without the three flags": auc_with(STATIC, COUNTS)})

## Unit 5: Is it memorizing, and the verdict

**Predict first.** Will the model's ROC AUC on the training part be more than 0.03 above its held-out AUC?

In [ ]:
print(round(model.score(X_train, y_train), 4), round(model.score(X_test, y_test), 4))
print(round(roc_auc_score(y_train, model.predict_proba(X_train)[:, 1]), 4), round(roc_auc_score(y_test, chance), 4))

## Finish it

### Exercise 1: Finish it

**Task:** **How good is the day-7 model on another split?** Split the people 80 to 20 with the seed 5 (given), fit the recipe of unit 2 on the training part only, and put the ROC AUC of its chances on the test part in `answer_1`, rounded to 3 decimals.

**Expected:** `answer_1` is a number between 0.5 and 1.

In [ ]:
X5_train, X5_test, y5_train, y5_test = train_test_split(X, y, test_size=0.2, random_state=5, stratify=y)    # given: an 80 to 20 split with seed 5
recipe = make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", StandardScaler(), WEEK1)]), LogisticRegression(max_iter=1000))    # given: the recipe of unit 2
# fit the recipe on the training part only, then compute the ROC AUC of the chances it gives the test part
answer_1 = None  # replace None with the AUC, rounded to 3 decimals

In [ ]:
_ = check("Exercise 1", "answer_1", 0.704, traps={0.69: "That is the AUC on the training part. Score the test part.", 0.643: "That scores the hard 0 and 1 calls. Give roc_auc_score the chances from predict_proba."}, tol=5e-4)

<details>
<summary>Hint</summary>

Fit with `recipe.fit(X5_train, y5_train)`. Then `recipe.predict_proba(X5_test)[:, 1]` gives the chance of churning for each test person, the call of unit 2 that `roc_auc_score` needs.

</details>

<details>
<summary>Half-solution</summary>

```python
recipe.fit(..., ...)
answer_1 = round(float(roc_auc_score(..., recipe.predict_proba(...)[:, 1])), 3)
```

</details>

<details>
<summary>Solution</summary>

```python
recipe.fit(X5_train, y5_train)
answer_1 = round(float(roc_auc_score(y5_test, recipe.predict_proba(X5_test)[:, 1])), 3)
```

On this split the recipe scores 0.704 on 3,564 test people. The check cell also names two near misses: the AUC on the training part (0.690) and the AUC of the hard 0 and 1 calls (0.643), which throws away the ranking. It is not the 0.690 of unit 2 because it is another split: Exercise 3 shows how far the score moves from split to split.

</details>

## On your own

### Exercise 2: On your own

**Task:** **Does the app's region help?** Before you start, say in one sentence what the result looks like: two numbers, the first between 0.65 and 0.75, the second very close to 0. The dataset README says region only decides the hour of day of a person's events and has no effect on anything else. Use the split and the recipe of Exercise 1. Refit the recipe without `region` and put the list [its ROC AUC on the test part, the full recipe's AUC minus that AUC] in `answer_2`, each rounded to 3 decimals.

**Expected:** `answer_2` is a list of two numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [0.705, -0.001], traps={(0.705, 0.001): "The second number is the full recipe's AUC minus the AUC without region, not the other way round."}, tol=5e-4)

<details>
<summary>Hint</summary>

Copy the recipe of Exercise 1 and take `"region"` out of the list of text columns; select the same columns from `X5_train` and `X5_test`. Fit the full recipe too, so that you can subtract.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"Two numbers: the AUC without region, about 0.70, and a difference of a few thousandths at most."*

```python
no_region = make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), [...]), ("num", StandardScaler(), WEEK1)]), LogisticRegression(max_iter=1000))
recipe.fit(X5_train, y5_train)
```

</details>

<details>
<summary>Solution</summary>

```python
recipe.fit(X5_train, y5_train)
no_region = make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["channel", "platform"]), ("num", StandardScaler(), WEEK1)]), LogisticRegression(max_iter=1000))
no_region.fit(X5_train[["channel", "platform"] + WEEK1], y5_train)
auc_no_region = roc_auc_score(y5_test, no_region.predict_proba(X5_test[["channel", "platform"] + WEEK1])[:, 1])
auc_full = roc_auc_score(y5_test, recipe.predict_proba(X5_test)[:, 1])
answer_2 = [round(float(auc_no_region), 3), round(float(auc_full - auc_no_region), 3)]
```

Without `region` the recipe scores 0.705; the full recipe scores 0.704. The difference, -0.001, is inside the noise of one split. The rules of the dataset say the same: region sets the hour of day of a person's events and nothing the model sees. A column that adds nothing on the test people is a candidate to stop defining; the decision itself belongs to a validation part.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague wants one number for the model's ROC AUC. The colleague scores the recipe on twenty different splits of the same people and reports the best of the twenty. Change the last line so that it reports a number you could defend, and put it in `answer_3`, rounded to 3 decimals.

**Expected:** `answer_3` is a number between 0.5 and 1.

In [ ]:
recipe3 = make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", StandardScaler(), WEEK1)]), LogisticRegression(max_iter=1000))   # given
scores = []
for seed in range(100, 120):                                           # given: twenty different splits of the same people
    a, b, c, d = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    scores.append(roc_auc_score(d, recipe3.fit(a, c).predict_proba(b)[:, 1]))
answer_3 = round(float(max(scores)), 3)        # colleague: "I tried twenty splits and report the best one"

In [ ]:
_ = check("Exercise 3", "answer_3", 0.691, traps={0.701: "That is the best split of the twenty. A number you can defend is one the splits agree on: the middle of the twenty."}, tol=5e-4)

<details>
<summary>Hint</summary>

If you could not look at the twenty scores, which single number would you say is typical? Think of the middle one, and `np.median`.

</details>

<details>
<summary>Half-solution</summary>

```python
answer_3 = round(float(np.median(...)), 3)
```

</details>

<details>
<summary>Solution</summary>

```python
answer_3 = round(float(np.median(scores)), 3)
```

The best of the twenty is 0.701 and the worst 0.684; the median is 0.691. Choosing the best split picks the luckiest draw, a score the model will not repeat on new people, and it hides how far the score moves. Report the middle with the range: AUC 0.691, 0.684 to 0.701 over twenty splits. The headline split of the lesson (seed 2026, 0.690) sits near that middle, which is why it was kept.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")